In [56]:
import re
import os

import pdfplumber

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None
    
import nbformat as nbf, os, textwrap

print("Ready")

Ready


In [57]:
def pdf_to_text(pdf_path: str, method: str = "pdfplumber") -> str:
    """
    Extract text from a text-native PDF.
    method: 'pdfplumber' or 'pymupdf'
    """
    if method == "pdfplumber":
        chunks = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                txt = page.extract_text() or ""
                txt = re.sub(r"\r\n?", "\n", txt)
                chunks.append(txt)
        return "\n\n".join(chunks)

    if method == "pymupdf":
        if fitz is None:
            raise RuntimeError("PyMuPDF not available")
        doc = fitz.open(pdf_path)
        return "\n\n".join(page.get_text("text") or "" for page in doc)

    raise ValueError(f"Unknown method: {method}")


In [58]:
# raw_text = pdf_to_text(PDF_PATH)
# print("chars:", len(raw_text))
# print(raw_text[:1000])

In [59]:
def extract_references_section(text: str) -> str:
    t = text.replace("\r\n", "\n").replace("\r", "\n")

    # 1) Prefer a line that starts with REFERENCES (even if more text follows)
    m = re.search(r"(?mi)^\s*references\b", t)
    if m:
        return t[m.start():].strip()

    # 2) Fallback: first occurrence anywhere (less ideal but better than failing)
    m = re.search(r"(?i)\breferences\b", t)
    if m:
        return t[m.start():].strip()

    raise ValueError("Could not find 'references' in the extracted text.")


In [60]:
def trim_to_first_reference(refs_text: str) -> str:
    t = refs_text.replace("\r\n", "\n").replace("\r", "\n")

    # Prefer: (1) followed by an author-like "Word," pattern soon after
    m = re.search(r"\(\s*1\s*\)\s*[A-Z][A-Za-z'’\-]+\s*,", t)
    if m:
        return t[m.start():].strip()

    # Fallback 1: any (1) anywhere
    m = re.search(r"\(\s*1\s*\)", t)
    if m:
        return t[m.start():].strip()

    raise ValueError("Couldn't find a '(1)' marker in references text.")


In [61]:
def normalize_for_splitting(text: str) -> str:
    # keep newlines; only safe repairs
    t = text.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)   # join hyphenated line breaks
    t = re.sub(r"[ \t]+", " ", t)           # collapse spaces/tabs
    return t.strip()

In [62]:
def split_references_acs_safe(text: str) -> list[str]:
    """
    Split on '(n)' markers that are likely true reference starts.
    Accept if preceded by start/whitespace/punctuation, but reject if preceded by a digit
    (avoids issue numbers like 53(19)).
    """
    t = text.replace("\r\n", "\n").replace("\r", "\n").strip()

    # Find all "(number)" occurrences
    matches = list(re.finditer(r"\(\s*\d+\s*\)", t))
    if not matches:
        raise ValueError("No '(n)' markers found.")

    starts = []
    for m in matches:
        i = m.start()
        prev = t[i-1] if i > 0 else ""
        # Reject issue numbers like 53(19)
        if prev.isdigit():
            continue
        # Accept if start or common separators
        if i == 0 or prev.isspace() or prev in ".;:,":
            starts.append(i)

    if not starts:
        raise ValueError("No safe reference-start markers found.")

    refs = []
    for idx, s in enumerate(starts):
        e = starts[idx + 1] if idx + 1 < len(starts) else len(t)
        refs.append(t[s:e].strip())

    return refs



In [63]:
def normalize_entry(entry: str) -> str:
    # flatten within-entry after splitting
    t = entry.replace("\r\n", "\n").replace("\r", "\n")
    t = re.sub(r"\s*\n\s*", " ", t)     # newlines -> spaces
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

In [64]:
PDF_PATH = "test.pdf"

raw = pdf_to_text(PDF_PATH)
refs = extract_references_section(raw)
refs = trim_to_first_reference(refs)
refs = normalize_for_splitting(refs)

entries = split_references_acs_line(refs)
entries = [normalize_entry(e) for e in entries]

print("entries:", len(entries))
print(entries[0][:600])


entries: 40
(1) Xiang, M.; Lyu, D.; Qin, Y.; Chen, R.; Liu, L.; Men, Y. https://pubs.acs.org/doi/10.1021/acsapm.5c04081. Microstructure of Bottlebrush Poly(n-Alkyl Methacrylate)s beyond Side Chain Packing. Polymer2020, 210,123034. Additional experimental details; Molecular weight


In [65]:
nums = []
for e in entries:
    m = re.match(r"\(\s*(\d+)\s*\)", e)
    if m:
        nums.append(int(m.group(1)))

print("min/max:", min(nums), max(nums), "count:", len(nums), "unique:", len(set(nums)))
missing = sorted(set(range(1, max(nums)+1)) - set(nums))
print("missing (first 25):", missing[:25])


min/max: 1 55 count: 40 unique: 38
missing (first 25): [3, 5, 6, 7, 10, 11, 15, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]


In [68]:
missing = [3, 5, 6, 7, 10, 11, 15, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45]

for n in missing:
    m = re.search(rf"\(\s*{n}\s*\)", refs)
    if not m:
        print(n, "NOT FOUND AT ALL")
        continue
    a = max(0, m.start()-80)
    b = min(len(refs), m.end()+120)
    print("\n#", n)
    print(refs[a:b].replace("\n", "\\n"))



# 3
ntrations (Figure S2); 8569.\nConfocal microscopy images of thermoresponsive PSA (3) Hempel, E.; Budde, H.; Höring, S.; Beiner, M. On the\nfilms stained with different dyes (Figures S3−S5); Crystallization

# 5
phiphilicComb\n■ Polymer Film. Langmuir2002, 18(8),2975−2979.\nAUTHOR INFORMATION (5) Zhang, M.; Estournes̀, C.; Bietsch, W.; Müller, A. H. E.\nCorresponding Author SuperparamagneticHybridNanocylinders.Adv

# 6
als2004,\n14(9), 871−882.\nEric J. Seibel − Department of Mechanical Engineering, (6)Miyake,G.M.;Weitekamp,R.A.;Piunova,V.A.;Grubbs,R.H.\nUniversityofWashington,Seattle,Washington98195,United Synthesis of I

# 7
lecting Photonic Crystals. J. Am.\nChem. Soc. 2012,134 (34),14249−14254.\nAuthors (7) Runge, M. B.; Bowden, N. B. Synthesis of High Molecular\nRuofan Liu − Department of Mechanical Engineering, Weight Comb 

# 10
d Engineering, 7526−7534.\nUniversityofWashington,Seattle,Washington98195,United (10)Zhou,J.;Turner,S.A.;Brosnan,S.M.;Li,Q.;Carrillo,J.-M.Y.;\nStat

In [67]:
entries = split_references_acs_safe(refs)
entries = [normalize_entry(e) for e in entries]

nums = []
for e in entries:
    m = re.match(r"\(\s*(\d+)\s*\)", e)
    nums.append(int(m.group(1)) if m else None)

print("entries:", len(entries))
print("min/max:", min(n for n in nums if n is not None), max(n for n in nums if n is not None))
print("unique:", len(set(n for n in nums if n is not None)))
missing = sorted(set(range(1, max(n for n in nums if n is not None)+1)) - set(n for n in nums if n is not None))
print("missing:", missing[:60])


entries: 68
min/max: 1 55
unique: 55
missing: []


In [69]:
def ref_number(entry: str):
    m = re.match(r"\(\s*(\d+)\s*\)", entry.strip())
    return int(m.group(1)) if m else None

def dedupe_keep_best(entries: list[str]) -> list[str]:
    buckets = {}
    for e in entries:
        n = ref_number(e)
        if n is None:
            continue
        buckets.setdefault(n, []).append(e)

    # pick best per n: longest length
    best = {}
    for n, lst in buckets.items():
        best[n] = max(lst, key=len)

    # return in numeric order
    return [best[n] for n in sorted(best.keys())]

entries_best = dedupe_keep_best(entries)

print("best entries:", len(entries_best))
print("numbers:", (ref_number(entries_best[0]), ref_number(entries_best[-1])))


best entries: 55
numbers: (1, 55)


In [70]:
for i in [1,2,3,36,55]:
    e = entries_best[i-1]
    print("\n---", i, "---")
    print(e[:500])



--- 1 ---
(1) Xiang, M.; Lyu, D.; Qin, Y.; Chen, R.; Liu, L.; Men, Y. https://pubs.acs.org/doi/10.1021/acsapm.5c04081. Microstructure of Bottlebrush Poly(n-Alkyl Methacrylate)s beyond Side Chain Packing. Polymer2020, 210,123034. Additional experimental details; Molecular weight

--- 2 ---
(2) Zhang, Z.; Chen, M.; Schneider, I.; Liu, Y.; Liang, S.; Sun, S.; characteristics of poly(n-alkyl acrylate) copolymers and Koynov, K.; Butt, H.-J.; Wu, S. Long Alkyl Side Chains LDT AH115 polymer (Tables S1, S2); Photo of Smart Simultaneously Improve Mechanical Robustness and Healing Ability substrate of VAHEAT system (Figure S1); TRA spatial ofaPhotoswitchablePolymer.Macromolecules2020,53(19),8562− distribution at different concentrations (Figure S2); 8569. Confocal microscopy images of ther

--- 3 ---
(3) Hempel, E.; Budde, H.; Höring, S.; Beiner, M. On the films stained with different dyes (Figures S3−S5); Crystallization Behavior of Frustrated Alkyl Groups in Poly(nContact angle and surface en

In [71]:
def strip_embedded_other_markers(entry: str) -> str:
    n = ref_number(entry)
    if n is None:
        return entry

    # Find any later "(k)" markers where k != n and cut before the first such marker
    for m in re.finditer(r"\(\s*(\d+)\s*\)", entry):
        k = int(m.group(1))
        if k != n and m.start() > 0:
            return entry[:m.start()].strip()

    return entry.strip()


In [78]:
# Truncate at these headings if they appear inside an entry
_TRUNC_HEADINGS = re.compile(
    r"(?i)\b(AUTHOR INFORMATION|ACKNOWLEDGMENTS|Funding|Notes|Complete contact information is available)\b"
)

# Remove these *spans* wherever they appear (don't drop whole line)
_SPAN_PATTERNS = [
    r"\bSupporting Information is available.*?(?=(\(\s*\d+\s*\))|$)",  # optional
    r"\bAdditional experimental details.*?(?=(\(\s*\d+\s*\))|$)",       # optional
    r"\bFigure\s*S\d+\b",
    r"\bFigures\s*S\d+(?:[–-]S\d+)?\b",
    r"\bTable\s*S\d+\b",
    r"\bTables\s*S\d+(?:\s*,\s*S\d+)*\b",
    r"\(PDF\)",
    r"\borcid\.org/\S+",
    r"\bEmail:\s*\S+",
    r"\bUniversityofWashington,Seattle,Washington98195,United\s*States\b",
    r"\bUniversity of Washington, Seattle, Washington 98195, United States\b",
    r"\bDepartment of [^.]*?(?=\.|;|,|$)",
]

_span_re = re.compile("|".join(_SPAN_PATTERNS), re.IGNORECASE)

def clean_entry(entry: str) -> str:
    t = entry.replace("\r\n", "\n").replace("\r", "\n")

    # 1) truncate at headings that should never be inside a reference
    m = _TRUNC_HEADINGS.search(t)
    if m:
        t = t[:m.start()]

    # 2) remove boilerplate spans (not whole lines)
    t = _span_re.sub(" ", t)

    # 3) collapse whitespace/newlines
    t = re.sub(r"\s+", " ", t).strip()

    return t


In [79]:
def ensure_marker(n: int, s: str) -> str:
    s = s.strip()
    if re.match(r"^\(\s*\d+\s*\)", s):
        return s
    return f"({n}) " + s

cleaned = []
for e in entries_best:
    n = ref_number(e)
    if n is None:
        # keep it for debugging; you can also skip
        cleaned.append(e)
        continue

    e1 = strip_embedded_other_markers(e)
    e2 = remove_boilerplate(e1)
    e3 = normalize_entry(e2)

    e3 = ensure_marker(n, e3)
    cleaned.append(e3)

# sanity check (ignore Nones just in case)
nums = [ref_number(e) for e in cleaned]
nums_ok = [n for n in nums if n is not None]

print("count:", len(cleaned),
      "unique:", len(set(nums_ok)),
      "min/max:", min(nums_ok), max(nums_ok),
      "missing:", sorted(set(range(1, max(nums_ok)+1)) - set(nums_ok))[:60])

# show any that lost markers
bad = [(i, cleaned[i]) for i,n in enumerate(nums) if n is None]
print("marker-missing entries:", len(bad))
for i, txt in bad[:3]:
    print("\n--- bad index", i, "---\n", txt[:400])


count: 55 unique: 55 min/max: 1 55 missing: []
marker-missing entries: 0


In [80]:
for i in [1,2,3,36,55]:
    print("\n---", i, "---")
    print(cleaned[i-1][:500])



--- 1 ---
(1) Xiang, M.; Lyu, D.; Qin, Y.; Chen, R.; Liu, L.; Men, Y. https://pubs.acs.org/doi/10.1021/acsapm.5c04081. Microstructure of Bottlebrush Poly(n-Alkyl Methacrylate)s beyond Side Chain Packing. Polymer2020, 210,123034. Additional experimental details; Molecular weight

--- 2 ---
(2) 

--- 3 ---
(3) 

--- 36 ---
(36) Ismael, R.; Schwander, H.; Hendrix, P. Fluorescent Dyes and MacKenzie,D.;Taroc,A.-M.;Gow,K.W.;Nelson,L.Y.;Seibel,E.J. Pigments. Ullmann’s Encycl.Ind.Chem. 2013, 1−22. A Temperature-Sensitive, High-Adhesion Medical Tape: A Com-

--- 55 ---
(55)Yusoh,N.A.;Gill,M.R.;Tian,X.AdvancingSuper-Resolution CrystallizationBehaviorofSemicrystallinePolymersCharacterizedby Microscopy with Metal Complexes: Functional Imaging Agents for an In Situ Fluorescence Technique and Its Sensing Mechanism. Nanoscale Visualization.Chem. Soc. Rev.2025, 54


In [ ]:
# def trim_to_first_reference(refs_text: str) -> str:
#     m = re.search(r"(?m)^\s*\(\s*1\s*\)\s+", refs_text)
#     if not m:
#         m = re.search(r"\(\s*1\s*\)\s+", refs_text)
#         if not m:
#             raise ValueError("Couldn't find the first reference marker (1).")
#     return refs_text[m.start():].strip()

# refs_only = trim_to_first_reference(refs_text)
# print(refs_only[:600])


In [20]:
# def split_references_acs_anywhere(text: str) -> list[str]:
#     """
#     Split ACS-style references on markers like '(1)', '(2)', ... even if they are not at line starts.
#     Uses finditer + slicing so it doesn't care about newlines.
#     """
#     t = text.strip()

#     # Find all occurrences of (number)
#     matches = list(re.finditer(r"\(\s*\d+\s*\)", t))
#     if not matches:
#         raise ValueError("No '(n)' reference markers found.")

#     refs = []
#     for i, m in enumerate(matches):
#         start = m.start()
#         end = matches[i + 1].start() if i + 1 < len(matches) else len(t)
#         chunk = t[start:end].strip()
#         refs.append(chunk)

#     return refs


In [23]:
# refs_norm = normalize_for_splitting(refs_only)
# entries = split_references_acs_line(refs_norm)

# print("entries:", len(entries))
# for i, e in enumerate(entries[:5], 1):
#     print("\n---", i, "---\n", e[:500])


entries: 40

--- 1 ---
 (1) Xiang, M.; Lyu, D.; Qin, Y.; Chen, R.; Liu, L.; Men, Y.
https://pubs.acs.org/doi/10.1021/acsapm.5c04081. Microstructure of Bottlebrush Poly(n-Alkyl Methacrylate)s beyond
Side Chain Packing. Polymer2020, 210,123034.
Additional experimental details; Molecular weight

--- 2 ---
 (2) Zhang, Z.; Chen, M.; Schneider, I.; Liu, Y.; Liang, S.; Sun, S.;
characteristics of poly(n-alkyl acrylate) copolymers and
Koynov, K.; Butt, H.-J.; Wu, S. Long Alkyl Side Chains
LDT AH115 polymer (Tables S1, S2); Photo of Smart
Simultaneously Improve Mechanical Robustness and Healing Ability
substrate of VAHEAT system (Figure S1); TRA spatial ofaPhotoswitchablePolymer.Macromolecules2020,53(19),8562−
distribution at different concentrations (Figure S2); 8569.
Confocal microscopy images of ther

--- 3 ---
 (4)Hyun,J.;Ma,H.;Banerjee,P.;Cole,J.;Gonsalves,K.;Chilkoti,
different temperatures (Figure S6) (PDF)
A.MicropatternsofaCell-AdhesivePeptideonanAmphiphilicComb
■ Polymer Film. Langmui

In [1]:
nb = nbf.v4.new_notebook()
cells = []

cells.append(nbf.v4.new_markdown_cell(textwrap.dedent("""
# PDF → DataFrame (companion to your Word bibliography notebook)

This notebook reads a **PDF** containing bibliography-style references and returns a `pandas.DataFrame`
in the **same schema** as your existing Word-based pipeline.

**Design intent**
- Reuse your existing parsing/normalization logic (the part that turns one reference string into one row).
- Only replace the *ingestion layer*: PDF → raw reference strings.
- Add debug cells so you can quickly tune splitting if the PDF formatting is quirky.

> Tip: If your Word notebook defines the parser function(s), we can load them with `%run bibliography.ipynb`.
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
# --- 0) Load your existing Word pipeline (recommended) ---
# This makes the PDF notebook schema-identical by reusing the same parsing function(s).

WORD_NOTEBOOK = "bibliography.ipynb"  # change if needed

try:
    get_ipython().run_line_magic("run", WORD_NOTEBOOK)
    print(f"Loaded: {WORD_NOTEBOOK}")
except Exception as e:
    print("Could not %run your Word notebook. That's OK, but you'll need to provide a parser function below.")
    print("Error:", repr(e))
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
import re
import pandas as pd

import pdfplumber

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
def pdf_to_text(pdf_path: str, method: str = "pdfplumber") -> str:
    \"\"\"Extract text from a text-native PDF.\"\"\"
    if method == "pdfplumber":
        chunks = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                txt = page.extract_text() or ""
                txt = re.sub(r"\\r\\n?", "\\n", txt)
                chunks.append(txt)
        return "\\n\\n".join(chunks)

    if method == "pymupdf":
        if fitz is None:
            raise RuntimeError("PyMuPDF not available; install pymupdf or use method='pdfplumber'")
        doc = fitz.open(pdf_path)
        chunks = [(page.get_text("text") or "") for page in doc]
        return "\\n\\n".join(chunks)

    raise ValueError(f"Unknown method: {method}")
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
def normalize_wrapped_lines(text: str) -> str:
    \"\"\"Fix common PDF text issues (hyphenation + wrapped lines).\"\"\"
    t = text

    # Join hyphenated line breaks: micro-\\nfluidics -> microfluidics
    t = re.sub(r"(\\w)-\\n(\\w)", r"\\1\\2", t)

    # Normalize newlines
    t = t.replace("\\r\\n", "\\n").replace("\\r", "\\n")

    # Protect paragraph breaks
    t = t.replace("\\n\\n", "\\uFFFF")

    # Convert remaining newlines to spaces
    t = re.sub(r"\\n+", " ", t)

    # Restore paragraph breaks
    t = t.replace("\\uFFFF", "\\n\\n")

    # Cleanup whitespace
    t = re.sub(r"[ \\t]+", " ", t)
    t = re.sub(r"\\n{3,}", "\\n\\n", t)

    return t.strip()
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
def split_references(text: str) -> list[str]:
    \"\"\"Split a bibliography into individual reference strings.\"\"\"
    t = text.strip()

    # [1] ... [2] ...
    if re.search(r"(?m)^\\s*\\[\\s*\\d+\\s*\\]\\s+", t):
        parts = re.split(r"(?m)^\\s*(\\[\\s*\\d+\\s*\\])\\s+", t)
        refs = []
        for i in range(1, len(parts), 2):
            label = parts[i]
            body = parts[i + 1] if i + 1 < len(parts) else ""
            refs.append((label + " " + body).strip())
        return [r for r in refs if r]

    # 1. ... 2. ...
    if re.search(r"(?m)^\\s*\\d+\\.\\s+", t):
        parts = re.split(r"(?m)^\\s*(\\d+)\\.\\s+", t)
        refs = []
        for i in range(1, len(parts), 2):
            n = parts[i]
            body = parts[i + 1] if i + 1 < len(parts) else ""
            refs.append((f"[{n}] " + body).strip())
        return [r for r in refs if r]

    # Fallback: paragraph split
    return [p.strip() for p in re.split(r"\\n\\s*\\n", t) if p.strip()]
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
# --- 1) Provide / select the parser that converts ONE reference string -> ONE row dict ---
#
# If your Word notebook defines a function like parse_reference_entry(...) or similar,
# set PARSE_FN to that function name.
#
# Examples (edit to match your code):
#   PARSE_FN = parse_reference_entry
#   PARSE_FN = parse_reference

PARSE_FN = None

# Heuristic: try to auto-detect a likely parser function name from globals.
_candidate_names = [
    "parse_reference_entry",
    "parse_reference",
    "reference_to_row",
    "parse_bibliography_entry",
    "parse_entry",
]
for _name in _candidate_names:
    if _name in globals() and callable(globals()[_name]):
        PARSE_FN = globals()[_name]
        print(f"Auto-selected PARSE_FN = {_name}")
        break

if PARSE_FN is None:
    print("PARSE_FN not set. Define it in this cell to match your Word pipeline schema.")
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
def pdf_bibliography_to_df(pdf_path: str, method: str = "pdfplumber") -> pd.DataFrame:
    if PARSE_FN is None:
        raise RuntimeError("PARSE_FN is not set. Edit the PARSE_FN cell to point to your existing parser.")

    raw_text = pdf_to_text(pdf_path, method=method)
    norm_text = normalize_wrapped_lines(raw_text)

    entries = split_references(norm_text)

    rows = []
    for entry in entries:
        rows.append(PARSE_FN(entry))

    return pd.DataFrame(rows)

# --- 2) Run ---
PDF_PATH = "test.pdf"  # change if needed
df_pdf = pdf_bibliography_to_df(PDF_PATH)
df_pdf.head()
""").strip()))

cells.append(nbf.v4.new_code_cell(textwrap.dedent("""
# --- Debug helpers ---
# Uncomment to inspect extraction quality and splitting.

# raw_text = pdf_to_text(PDF_PATH)
# print(raw_text[:2000])

# norm_text = normalize_wrapped_lines(raw_text)
# entries = split_references(norm_text)
# print("Entries:", len(entries))
# for i, e in enumerate(entries[:5], 1):
#     print("\\n---", i, "---\\n", e[:800])
""").strip()))

nb["cells"] = cells

out_path = "/mnt/data/pdf_bibliography_to_df.ipynb"
os.makedirs("/mnt/data", exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    nbf.write(nb, f)

out_path



'/mnt/data/pdf_bibliography_to_df.ipynb'

In [2]:
cells

[{'id': 'fb2a2da7',
  'cell_type': 'markdown',
  'source': '# PDF → DataFrame (companion to your Word bibliography notebook)\n\nThis notebook reads a **PDF** containing bibliography-style references and returns a `pandas.DataFrame`\nin the **same schema** as your existing Word-based pipeline.\n\n**Design intent**\n- Reuse your existing parsing/normalization logic (the part that turns one reference string into one row).\n- Only replace the *ingestion layer*: PDF → raw reference strings.\n- Add debug cells so you can quickly tune splitting if the PDF formatting is quirky.\n\n> Tip: If your Word notebook defines the parser function(s), we can load them with `%run bibliography.ipynb`.',
  'metadata': {}},
 {'id': 'b3c2a9ac',
  'cell_type': 'code',
  'metadata': {},
  'execution_count': None,
  'source': '# --- 0) Load your existing Word pipeline (recommended) ---\n# This makes the PDF notebook schema-identical by reusing the same parsing function(s).\n\nWORD_NOTEBOOK = "bibliography.ipynb"

In [4]:
import os

PDF_PATH = "test.pdf"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(PDF_PATH))
print("files here (pdf/zip):", [f for f in os.listdir(".") if f.lower().endswith((".pdf",".zip"))])


cwd: C:\Users\robin\Programs\KilbrethsPig
exists: True
files here (pdf/zip): ['Kilbreths_Pig_Report.pdf', 'test.pdf']


In [5]:
raw_text = pdf_to_text("test.pdf")   # pdfplumber default
print("chars:", len(raw_text))
print(raw_text[:1200])


NameError: name 'pdf_to_text' is not defined